# Original YOLOv8 baselines: VOC and KITTI

This notebook evaluates the four FP32 `.pt` models under `original/` with one reproducible pipeline. It reports **mAP50, mAP75, mAP50-95, Precision, Recall, Parameters, GFLOPs, Size (MB), Mean Latency (ms), P95 (ms), and FPS**.

Accuracy is computed by Ultralytics on each dataset's complete validation split (VOC2007 test and KITTI val). Precision and Recall are the class means reported by Ultralytics. Latency is batch-1 end-to-end `predict()` time (pre-processing + inference + NMS, excluding disk I/O) over preloaded validation images. FPS is `1000 / mean latency`. Results therefore describe the selected device and software environment, which are recorded with the CSV output.

In [ ]:
# Run once in a fresh environment, then restart the kernel if requested.
%pip install -q "ultralytics>=8.3" pandas pillow pyyaml

In [ ]:
from pathlib import Path
from time import perf_counter
import json
import platform

import numpy as np
import pandas as pd
import torch
import ultralytics
from PIL import Image
from IPython.display import display
from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils.torch_utils import get_flops

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'original').is_dir() and (candidate / 'datasets').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('Could not find a project folder containing original/ and datasets/.')

PROJECT_ROOT = find_project_root()
DATASETS_DIR = PROJECT_ROOT / 'datasets'
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'baseline_benchmark'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
settings.update({'datasets_dir': str(DATASETS_DIR)})

IMGSZ = 640
VAL_BATCH = 16
VAL_WORKERS = 0             # portable and notebook-safe
LATENCY_IMAGES = 100        # set to None to time the full validation split
LATENCY_WARMUP = 10
LATENCY_CONF = 0.25
LATENCY_IOU = 0.70
SEED = 0

if torch.cuda.is_available():
    DEVICE = 0
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

MODELS = [
    {'model': 'YOLOv8n', 'dataset': 'VOC',   'weights': PROJECT_ROOT / 'original/yolov8n_voc/final_model_factorized_lwi.pt', 'data': DATASETS_DIR / 'VOC/VOC.yaml'},
    {'model': 'YOLOv8l', 'dataset': 'VOC',   'weights': PROJECT_ROOT / 'original/yolo8l_voc/voc_8l_best.pt',             'data': DATASETS_DIR / 'VOC/VOC.yaml'},
    {'model': 'YOLOv8n', 'dataset': 'KITTI', 'weights': PROJECT_ROOT / 'original/yolov8n_kitti/final_model_factorized_lwi.pt', 'data': DATASETS_DIR / 'kitti/kitti.yaml'},
    {'model': 'YOLOv8l', 'dataset': 'KITTI', 'weights': PROJECT_ROOT / 'original/yolov8l_kitti/8l_float32.pt',               'data': DATASETS_DIR / 'kitti/kitti.yaml'},
]

print(f'Project: {PROJECT_ROOT}')
print(f'Device:  {DEVICE}')
print(f'PyTorch {torch.__version__} | Ultralytics {ultralytics.__version__}')

In [ ]:
# Fail early if a weight, YAML, or validation split is missing.
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

def validation_images(data_yaml):
    info = check_det_dataset(str(data_yaml), autodownload=False)
    locations = info['val'] if isinstance(info['val'], list) else [info['val']]
    images = []
    for location in map(Path, locations):
        if location.is_dir():
            images.extend(p for p in location.rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)
        elif location.is_file() and location.suffix.lower() == '.txt':
            for line in location.read_text().splitlines():
                path = Path(line.strip())
                images.append(path if path.is_absolute() else (location.parent / path).resolve())
        elif location.suffix.lower() in IMAGE_SUFFIXES:
            images.append(location)
    images = sorted(set(map(Path, images)))
    if not images:
        raise FileNotFoundError(f'No validation images resolved from {data_yaml}')
    return images

inventory = []
for item in MODELS:
    if not item['weights'].is_file():
        raise FileNotFoundError(item['weights'])
    if not item['data'].is_file():
        raise FileNotFoundError(item['data'])
    count = len(validation_images(item['data']))
    inventory.append({
        'Model': item['model'], 'Dataset': item['dataset'],
        'Validation images': count,
        'Weights': str(item['weights'].relative_to(PROJECT_ROOT)),
    })
display(pd.DataFrame(inventory))

In [ ]:
def synchronize(device):
    device = str(device)
    if device.startswith('cuda') or (device.isdigit() and torch.cuda.is_available()):
        torch.cuda.synchronize()
    elif device == 'mps' and torch.backends.mps.is_available():
        torch.mps.synchronize()

def model_statistics(yolo, weights, imgsz=640):
    core = yolo.model
    parameters = sum(parameter.numel() for parameter in core.parameters())
    try:
        gflops = float(get_flops(core, imgsz=imgsz))
    except Exception as exc:
        print(f'GFLOPs calculation warning: {exc}')
        gflops = float('nan')
    return parameters, gflops, weights.stat().st_size / 1_000_000

def benchmark_latency(yolo, image_paths, device, imgsz=640, warmup=10, limit=100):
    selected = image_paths if limit is None else image_paths[:min(limit, len(image_paths))]
    # Preload images so storage speed is not part of the reported latency.
    images = []
    for path in selected:
        with Image.open(path) as image:
            images.append(np.asarray(image.convert('RGB')).copy())

    for index in range(warmup):
        yolo.predict(source=images[index % len(images)], imgsz=imgsz, device=device, conf=LATENCY_CONF, iou=LATENCY_IOU, verbose=False, save=False)
    synchronize(device)

    elapsed_ms = []
    for image in images:
        synchronize(device)
        started = perf_counter()
        yolo.predict(source=image, imgsz=imgsz, device=device, conf=LATENCY_CONF, iou=LATENCY_IOU, verbose=False, save=False)
        synchronize(device)
        elapsed_ms.append((perf_counter() - started) * 1000.0)

    values = np.asarray(elapsed_ms, dtype=np.float64)
    mean_ms = float(values.mean())
    return {'Mean Latency (ms)': mean_ms, 'P95 (ms)': float(np.percentile(values, 95)), 'FPS': 1000.0 / mean_ms, 'Latency Samples': len(values)}

def evaluate_one(spec):
    print(f"\n=== {spec['model']} / {spec['dataset']} ===")
    yolo = YOLO(str(spec['weights']))
    parameters, gflops, size_mb = model_statistics(yolo, spec['weights'], IMGSZ)
    metrics = yolo.val(
        data=str(spec['data']), split='val', imgsz=IMGSZ, batch=VAL_BATCH, device=DEVICE,
        workers=VAL_WORKERS, plots=False, save_json=False, verbose=False,
        project=str(OUTPUT_DIR / 'validation'), name=f"{spec['dataset'].lower()}_{spec['model'].lower()}", exist_ok=True,
    )
    latency = benchmark_latency(yolo, validation_images(spec['data']), DEVICE, IMGSZ, LATENCY_WARMUP, LATENCY_IMAGES)
    row = {
        'Model': spec['model'], 'Dataset': spec['dataset'],
        'mAP50': float(metrics.box.map50), 'mAP75': float(metrics.box.map75), 'mAP50-95': float(metrics.box.map),
        'Precision': float(metrics.box.mp), 'Recall': float(metrics.box.mr),
        'Parameters': int(parameters), 'GFLOPs': gflops, 'Size (MB)': size_mb, **latency,
        'Device': str(DEVICE), 'Image Size': IMGSZ, 'Weights': str(spec['weights'].relative_to(PROJECT_ROOT)),
    }
    del yolo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row

In [ ]:
# Full evaluation: four models across their corresponding VOC/KITTI validation split.
RESULTS_CSV = OUTPUT_DIR / 'baseline_results.csv'
ENVIRONMENT_JSON = OUTPUT_DIR / 'environment.json'
rows, errors = [], {}
for spec in MODELS:
    key = f"{spec['dataset']}_{spec['model']}"
    try:
        rows.append(evaluate_one(spec))
        pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
    except Exception as exc:
        errors[key] = f'{type(exc).__name__}: {exc}'
        print(f'FAILED {key}: {errors[key]}')

environment = {
    'platform': platform.platform(), 'processor': platform.processor(), 'python': platform.python_version(),
    'torch': torch.__version__, 'ultralytics': ultralytics.__version__, 'device': str(DEVICE),
    'image_size': IMGSZ, 'validation_batch': VAL_BATCH, 'latency_warmup': LATENCY_WARMUP,
    'latency_images': LATENCY_IMAGES,
    'latency_definition': 'batch-1 predict; preprocess + inference + NMS; disk I/O excluded',
    'errors': errors,
}
ENVIRONMENT_JSON.write_text(json.dumps(environment, indent=2))
results = pd.DataFrame(rows)
if results.empty:
    raise RuntimeError(f'All evaluations failed: {errors}')
display(results)
if errors:
    print('Errors:', json.dumps(errors, indent=2))
print(f'\nSaved: {RESULTS_CSV}')
print(f'Saved: {ENVIRONMENT_JSON}')

In [ ]:
# Dissertation-friendly formatted table.
metric_columns = ['Model', 'Dataset', 'mAP50', 'mAP75', 'mAP50-95', 'Precision', 'Recall', 'Parameters', 'GFLOPs', 'Size (MB)', 'Mean Latency (ms)', 'P95 (ms)', 'FPS']
formatted = results[metric_columns].sort_values(['Dataset', 'Model']).reset_index(drop=True)
display(formatted.style.format({
    'mAP50': '{:.4f}', 'mAP75': '{:.4f}', 'mAP50-95': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}',
    'Parameters': '{:,}', 'GFLOPs': '{:.2f}', 'Size (MB)': '{:.2f}',
    'Mean Latency (ms)': '{:.2f}', 'P95 (ms)': '{:.2f}', 'FPS': '{:.2f}',
}).set_caption(f'YOLOv8 baselines at {IMGSZ} px on {DEVICE}'))